# Projekt - Języki Naturalne
## Autorzy:
- Jan Rapacz
- Piotr Skowroński
- Jakub Małecki
- Jakub Pietrasik

## Cel
Celem projektu jest wykrywanie wybranych struktur kombinatorycznych w danych pochodzących z języków naturalnych oraz ocena, jak często i w jakiej postaci pojawiają się one w rzeczywistym materiale językowym.

## Przygotowanie środowiska

Importujemy detektory oraz definiujemy małą funkcję pomocniczą, która zamienia obiekty `StructureMatch` na czytelne wiersze. Dzięki temu wyniki `find(...)` są łatwiejsze do pokazania podczas prezentacji.

In [14]:
import sys
from pathlib import Path

try:
    from IPython import get_ipython
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    get_ipython = lambda: None
    display = print


def find_project_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "natural_languages").is_dir():
            return path
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from natural_languages.detectors import (  # noqa: E402
    AbelianSquareDetector,
    AnagramDetector,
    PalindromeDetector,
    ShuffledSquareDetector,
    SquareDetector,
    TangramDetector,
)


def render_markdown(text):
    if Markdown is None or get_ipython() is None:
        print(text)
    else:
        display(Markdown(text))


def format_bool(value):
    return "tak" if value else "nie"


def escape_cell(value):
    return str(value).replace("|", "\\|").replace("\n", "<br>")


def markdown_table(headers, rows):
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join("---" for _ in headers) + " |"
    body = [
        "| " + " | ".join(escape_cell(row.get(header, "")) for header in headers) + " |"
        for row in rows
    ]
    return "\n".join([header, separator, *body])


def display_table(title, headers, rows):
    if rows:
        render_markdown(f"### {title}\n\n{markdown_table(headers, rows)}")
    else:
        render_markdown(f"### {title}\n\nBrak wyników.")


def check_rows(words, check):
    return [{"słowo": word or "(puste)", "wynik": format_bool(check(word))} for word in words]


def match_rows(matches, limit=12):
    rows = [
        {
            "typ": match.structure_type,
            "fragment": match.word,
            "zakres": f"[{match.start}, {match.end})",
            "części": " + ".join(match.parts),
        }
        for match in matches[:limit]
    ]
    if len(matches) > limit:
        rows.append(
            {
                "typ": "...",
                "fragment": f"pokazano {limit} z {len(matches)} wyników",
                "zakres": "",
                "części": "",
            }
        )
    return rows


def display_matches(title, matches, limit=12):
    display_table(title, ["typ", "fragment", "zakres", "części"], match_rows(matches, limit))


squares = SquareDetector()
palindromes = PalindromeDetector(min_length=2)
abelian = AbelianSquareDetector()
anagrams = AnagramDetector()
tangrams = TangramDetector()
shuffled = ShuffledSquareDetector()

render_markdown("**Detektory gotowe.**")

**Detektory gotowe.**

## 1. Kwadraty i sześciany

Kwadrat to słowo postaci `ww`, czyli dwie identyczne kopie tego samego bloku. Przykład: `kankan = kan + kan`.

W module `squares.py` klasa `SquareDetector` udostępnia:

- `check(word)` - sprawdza, czy całe słowo jest kwadratem,
- `find(word)` - znajduje wszystkie fragmenty będące kwadratami,
- `find_cubes(word)` - znajduje sześciany postaci `www`.

In [15]:
square_examples = ["kankan", "czacza", "abab", "kot", "abcabd"]

display_table(
    "check(word): czy całe słowo jest kwadratem",
    ["słowo", "wynik"],
    check_rows(square_examples, squares.check),
)
display_matches("find('kankan'): kwadraty znalezione w słowie", squares.find("kankan"))
display_matches("find_cubes('ababab'): sześciany znalezione w słowie", squares.find_cubes("ababab"))

### check(word): czy całe słowo jest kwadratem

| słowo | wynik |
| --- | --- |
| kankan | tak |
| czacza | tak |
| abab | tak |
| kot | nie |
| abcabd | nie |

### find('kankan'): kwadraty znalezione w słowie

| typ | fragment | zakres | części |
| --- | --- | --- | --- |
| square | kankan | [0, 6) | kan + kan |

### find_cubes('ababab'): sześciany znalezione w słowie

| typ | fragment | zakres | części |
| --- | --- | --- | --- |
| cube | ababab | [0, 6) | ab + ab + ab |

## 2. Palindromy

Palindrom to słowo równe swojemu odwróceniu. Przykłady: `kajak`, `anna`, `kayak`.

`PalindromeDetector` ma parametr `min_length`, który określa minimalną długość fragmentów zwracanych przez `find(...)`. W tej prezentacji używamy `min_length=2`, żeby pominąć trywialne palindromy jednoznakowe.

In [16]:
palindrome_examples = ["kajak", "anna", "kayak", "tatar", "kot"]

display_table(
    "check(word): czy całe słowo jest palindromem",
    ["słowo", "wynik"],
    check_rows(palindrome_examples, palindromes.check),
)
display_matches("find('abacaba'): palindromy znalezione w słowie", palindromes.find("abacaba"))

### check(word): czy całe słowo jest palindromem

| słowo | wynik |
| --- | --- |
| kajak | tak |
| anna | tak |
| kayak | tak |
| tatar | nie |
| kot | nie |

### find('abacaba'): palindromy znalezione w słowie

| typ | fragment | zakres | części |
| --- | --- | --- | --- |
| palindrome | aba | [0, 3) | aba |
| palindrome | abacaba | [0, 7) | abacaba |
| palindrome | bacab | [1, 6) | bacab |
| palindrome | aca | [2, 5) | aca |
| palindrome | aba | [4, 7) | aba |

## 3. Kwadraty abelowe

Kwadrat abelowy to słowo postaci `w1w2`, gdzie obie połowy mają tę samą długość i ten sam multizbiór liter. Kolejność liter w połowach nie musi być taka sama.

Przykład: `abba = ab + ba`. To nie jest zwykły kwadrat, ale jest kwadratem abelowym, bo `ab` i `ba` są anagramami.

In [17]:
abelian_examples = ["kryptoportyk", "abba", "kankan", "abcd", "abc"]

display_table(
    "check(word): czy całe słowo jest kwadratem abelowym",
    ["słowo", "wynik"],
    check_rows(abelian_examples, abelian.check),
)
display_matches("find('abba'): kwadraty abelowe znalezione w słowie", abelian.find("abba"))

### check(word): czy całe słowo jest kwadratem abelowym

| słowo | wynik |
| --- | --- |
| kryptoportyk | tak |
| abba | tak |
| kankan | tak |
| abcd | nie |
| abc | nie |

### find('abba'): kwadraty abelowe znalezione w słowie

| typ | fragment | zakres | części |
| --- | --- | --- | --- |
| abelian_square | abba | [0, 4) | ab + ba |
| abelian_square | bb | [1, 3) | b + b |

## 4. Anagramy

Anagramy to słowa o tych samych częstościach liter. W przeciwieństwie do poprzednich struktur, anagramy są relacją między co najmniej dwoma słowami, a nie strukturą jednego fragmentu.

`AnagramDetector` udostępnia:

- `check(word1, word2)` - sprawdza jedną parę słów,
- `find_groups(words)` - grupuje słowa będące wzajemnymi anagramami.

In [ ]:
anagram_words = ["kot", "tok", "pies", "psie", "dom", "mod", "odm", "las"]

display_table(
    "check(word1, word2): porównanie par słów",
    ["para", "wynik"],
    [
        {"para": "kot / tok", "wynik": format_bool(anagrams.check("kot", "tok"))},
        {"para": "kot / kos", "wynik": format_bool(anagrams.check("kot", "kos"))},
        {"para": "elevenplustwo / twelveplusone", "wynik": format_bool(anagrams.check("elevenplustwo", "twelveplusone"))}, 
    ],
)

anagram_groups = anagrams.find_groups(anagram_words)
display_table(
    "find_groups(...): znalezione grupy anagramów",
    ["nr", "słowa w grupie"],
    [
        {"nr": index, "słowa w grupie": ", ".join(group)}
        for index, group in enumerate(anagram_groups, start=1)
    ],
)

### check(word1, word2): porównanie par słów

| para | wynik |
| --- | --- |
| kot / tok | tak |
| kot / kos | nie |
| elevenplustwo / twelveplusone | tak |

### find_groups(...): znalezione grupy anagramów

| nr | słowa w grupie |
| --- | --- |
| 1 | kot, tok |
| 2 | pies, psie |
| 3 | dom, mod, odm |

## 5. Tangramy

W tym projekcie tangramem nazywamy słowo, w którym każda litera występuje parzystą liczbę razy.

To ważny warunek konieczny dla przetasowanych kwadratów: jeśli słowo da się rozłożyć na dwa identyczne podciągi, to każda litera musi występować parzyście. Warunek nie jest jednak wystarczający.

In [19]:
tangram_examples = ["aabbcc", "abccba", "kankan", "abc", ""]

display_table(
    "check(word): czy całe słowo jest tangramem",
    ["słowo", "wynik"],
    check_rows(tangram_examples, tangrams.check),
)
display_matches("find('aabbcc'): wynik detekcji tangramu", tangrams.find("aabbcc"))

### check(word): czy całe słowo jest tangramem

| słowo | wynik |
| --- | --- |
| aabbcc | tak |
| abccba | tak |
| kankan | tak |
| abc | nie |
| (puste) | nie |

### find('aabbcc'): wynik detekcji tangramu

| typ | fragment | zakres | części |
| --- | --- | --- | --- |
| tangram | aabbcc | [0, 6) | aabbcc |

## 6. Przetasowane kwadraty

Przetasowany kwadrat to słowo, które można rozłożyć na dwa identyczne podciągi. Podciągi nie muszą być spójnymi fragmentami, ale zachowują kolejność liter.

Przykład: `prepress` jest przetasowanym kwadratem, bo można w nim wskazać dwa takie same podciągi `pres`.

Detekcja jest kosztowna obliczeniowo, dlatego `ShuffledSquareDetector` ogranicza długość słowa do `MAX_WORD_LENGTH = 20`.

In [20]:
shuffled_examples = ["abab", "prepress", "abccba", "abcabd", "kot", ""]

render_markdown(f"**Limit długości:** `{shuffled.MAX_WORD_LENGTH}` znaków.")
display_table(
    "check(word): czy całe słowo jest przetasowanym kwadratem",
    ["słowo", "wynik"],
    check_rows(shuffled_examples, shuffled.check),
)
display_matches("find('prepress'): wynik detekcji przetasowanego kwadratu", shuffled.find("prepress"))

**Limit długości:** `20` znaków.

### check(word): czy całe słowo jest przetasowanym kwadratem

| słowo | wynik |
| --- | --- |
| abab | tak |
| prepress | tak |
| abccba | nie |
| abcabd | nie |
| kot | nie |
| (puste) | nie |

### find('prepress'): wynik detekcji przetasowanego kwadratu

| typ | fragment | zakres | części |
| --- | --- | --- | --- |
| shuffled_square | prepress | [0, 8) | prepress |

## 7. Porównanie na wspólnych przykładach

Na końcu sprawdzamy kilka tych samych słów różnymi detektorami. To dobrze pokazuje różnice między strukturami.

Najważniejsze obserwacje:

- każdy zwykły kwadrat jest też kwadratem abelowym,
- każdy przetasowany kwadrat musi być tangramem,
- tangram nie musi być przetasowanym kwadratem,
- anagramy są osobnym przypadkiem, bo porównują pary lub grupy słów.

In [22]:
demo_words = ["kankan", "kajak", "abba", "kryptoportyk", "prepress", "abccba", "kot"]

comparison = []
for word in demo_words:
    comparison.append(
        {
            "słowo": word,
            "kwadrat": format_bool(squares.check(word)),
            "palindrom": format_bool(palindromes.check(word)),
            "kwadrat abelowy": format_bool(abelian.check(word)),
            "tangram": format_bool(tangrams.check(word)),
            "przetasowany kwadrat": format_bool(shuffled.check(word)),
        }
    )

display_table(
    "Porównanie detektorów na tych samych słowach",
    ["słowo", "kwadrat", "palindrom", "kwadrat abelowy", "tangram", "przetasowany kwadrat"],
    comparison,
)

### Porównanie detektorów na tych samych słowach

| słowo | kwadrat | palindrom | kwadrat abelowy | tangram | przetasowany kwadrat |
| --- | --- | --- | --- | --- | --- |
| kankan | tak | nie | tak | tak | tak |
| kajak | nie | tak | nie | nie | nie |
| abba | nie | tak | tak | tak | nie |
| kryptoportyk | nie | nie | tak | tak | nie |
| prepress | nie | nie | nie | tak | tak |
| abccba | nie | tak | tak | tak | nie |
| kot | nie | nie | nie | nie | nie |